# Stage 1 — Data Construction

Builds harmful/harmless contrastive prompt pairs and the benign-but-adjacent counterfactual set, then splits into probe-train / probe-test / steering-construction / recovery-evaluation.

Sources:
- **Harmful base:** `CohereLabs/aya_redteaming` (English subset) — reused from the prior hackathon pipeline.
- **Harmful volume:** `walledai/AdvBench`.
- **Harmless:** `tatsu-lab/alpaca`, sampled to match count.
- **Benign-but-adjacent (Stage 6 counterfactual control):** `walledai/XSTest`, safe-labeled subset.

No GPU needed for this stage, but follows the same `claude.md` setup pattern for consistency. `/data` outputs get committed to GitHub (they're small, versioned inputs, not results).

In [ ]:
!pip install -q -U datasets huggingface_hub pandas scikit-learn

## 1. Mount Drive (claude.md rule 1)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/RepEng_Survival/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

## 2. Clone/pull repo (claude.md rule 1)

In [ ]:
REPO_URL = 'https://github.com/Pranamya16/RepEng_Survival.git'

if not os.path.exists('/content/RepEng_Survival'):
    !git clone {REPO_URL} /content/RepEng_Survival
else:
    !cd /content/RepEng_Survival && git pull

%cd /content/RepEng_Survival

import sys
sys.path.append('/content/RepEng_Survival/src')

DATA_DIR = '/content/RepEng_Survival/data'
os.makedirs(DATA_DIR, exist_ok=True)

## 3. Anti-disconnect (claude.md rule 2)

In [ ]:
from IPython.display import Javascript

display(Javascript('''
function KeepAlive(){
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(KeepAlive, 60000)
'''))

## 4. Hugging Face login
Add your token as a Colab secret named `HF_TOKEN` (Settings key icon in the left sidebar).

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get('HF_TOKEN'))

## 5. Load harmful prompts
`aya_redteaming` (English subset, reused from the hackathon pipeline) + `AdvBench` (volume extension).

In [ ]:
from datasets import load_dataset

aya = load_dataset('CohereLabs/aya_redteaming', split='aya_eng')
aya_harmful = [row['prompt'] for row in aya]
print(f'aya_redteaming (English): {len(aya_harmful)} prompts')

advbench = load_dataset('walledai/AdvBench', split='train')
advbench_harmful = [row['prompt'] for row in advbench]
print(f'AdvBench: {len(advbench_harmful)} prompts')

harmful_raw = aya_harmful + advbench_harmful
harmful_sources = ['aya_redteaming'] * len(aya_harmful) + ['advbench'] * len(advbench_harmful)

## 6. Dedupe harmful prompts

In [ ]:
seen = set()
harmful_prompts = []
harmful_prompt_sources = []
for prompt, source in zip(harmful_raw, harmful_sources):
    key = prompt.strip().lower()
    if key in seen or not key:
        continue
    seen.add(key)
    harmful_prompts.append(prompt.strip())
    harmful_prompt_sources.append(source)

print(f'Deduped harmful prompts: {len(harmful_prompts)}')

## 7. Load harmless counterparts (Alpaca, sampled to match)

In [ ]:
import random
random.seed(42)

alpaca = load_dataset('tatsu-lab/alpaca', split='train')
alpaca_no_input = [row['instruction'] for row in alpaca if not row['input']]

harmless_prompts = random.sample(alpaca_no_input, len(harmful_prompts))
harmless_prompt_sources = ['alpaca'] * len(harmless_prompts)

print(f'Harmless prompts (matched count): {len(harmless_prompts)}')

## 8. Load benign-but-adjacent set (XSTest, safe-labeled subset)
Used only in Stage 6 (counterfactual control) — prompts that sound risky but aren't.

In [ ]:
xstest = load_dataset('walledai/XSTest', split='test')

benign_adjacent_prompts = [row['prompt'] for row in xstest if row['label'].strip().lower() == 'safe']
print(f'XSTest safe (benign-but-adjacent): {len(benign_adjacent_prompts)} prompts')

## 9. Build splits
Non-overlapping probe-train / probe-test / steering-construction / recovery-evaluation splits, same indices used for both harmful and harmless pools so pairs stay aligned.

In [ ]:
from sklearn.model_selection import train_test_split

n = len(harmful_prompts)
indices = list(range(n))

train_idx, temp_idx = train_test_split(indices, test_size=0.6, random_state=42)
test_idx, temp_idx2 = train_test_split(temp_idx, test_size=2/3, random_state=42)
steer_idx, recovery_idx = train_test_split(temp_idx2, test_size=0.5, random_state=42)

SPLITS = {
    'probe_train': train_idx,
    'probe_test': test_idx,
    'steering_construction': steer_idx,
    'recovery_evaluation': recovery_idx,
}

for name, idx in SPLITS.items():
    print(f'{name}: {len(idx)}')

## 10. Save splits and benign-adjacent set to /data (versioned in GitHub)

In [ ]:
import json

def write_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

for split_name, idx in SPLITS.items():
    rows = []
    for i in idx:
        rows.append({'prompt': harmful_prompts[i], 'label': 'harmful', 'source': harmful_prompt_sources[i]})
        rows.append({'prompt': harmless_prompts[i], 'label': 'harmless', 'source': harmless_prompt_sources[i]})
    random.shuffle(rows)
    write_jsonl(os.path.join(DATA_DIR, f'{split_name}.jsonl'), rows)
    print(f'Wrote {split_name}.jsonl: {len(rows)} rows')

benign_adjacent_rows = [{'prompt': p, 'label': 'benign_adjacent', 'source': 'xstest'} for p in benign_adjacent_prompts]
write_jsonl(os.path.join(DATA_DIR, 'benign_adjacent.jsonl'), benign_adjacent_rows)
print(f'Wrote benign_adjacent.jsonl: {len(benign_adjacent_rows)} rows')

## 11. Write data card

In [ ]:
data_card = f'''# Data Card — Stage 1

## Sources
- Harmful base: CohereLabs/aya_redteaming (English subset) — {len(aya_harmful)} raw prompts
- Harmful volume: walledai/AdvBench — {len(advbench_harmful)} raw prompts
- Harmless: tatsu-lab/alpaca (no-input instructions, random sample matched to harmful count)
- Benign-but-adjacent: walledai/XSTest, safe-labeled subset — {len(benign_adjacent_prompts)} prompts

## Processing
- Harmful prompts deduped (case-insensitive) across aya_redteaming + AdvBench: {len(harmful_prompts)} unique prompts.
- Harmless prompts sampled 1:1 to match harmful count, seed=42.
- Split with sklearn train_test_split, seed=42, same indices applied to harmful and harmless pools so pairs stay aligned.

## Splits (harmful + harmless combined, shuffled)
'''
for split_name, idx in SPLITS.items():
    data_card += f'- {split_name}.jsonl: {len(idx) * 2} rows ({len(idx)} harmful + {len(idx)} harmless)\n'
data_card += '- benign_adjacent.jsonl: used only in Stage 6, never seen during probe/steering fitting\n'
data_card += '\n## Notes\n- recovery_evaluation split must never be used during probe/steering construction (Stage 2-5) — held out for Stage 5 recovery testing only, per project_plan.md.\n'

with open(os.path.join(DATA_DIR, 'DATA_CARD.md'), 'w', encoding='utf-8') as f:
    f.write(data_card)

print(data_card)

## 12. Push /data to GitHub
`/data` is versioned in the repo (small JSONL files, not results) — this is the one folder besides `/src` and `/notebooks` that gets pushed.

In [ ]:
!git config user.email "pranamya@pskulkarni.com"
!git config user.name "Pranamya16"
!cd /content/RepEng_Survival && git add data/ && git commit -m "Stage 1: data construction (aya_redteaming + AdvBench + alpaca + XSTest splits)" && git push

## Deliverable check
- [ ] `/data` has probe_train / probe_test / steering_construction / recovery_evaluation / benign_adjacent JSONL files
- [ ] `DATA_CARD.md` written with source counts and split sizes
- [ ] Pushed to GitHub